In [1]:
import numpy as np
import pandas as pd
import sklearn as skl
import streamlit as sl
import pickle as pkl
import ast

In [2]:
movieDf=pd.read_csv("Dataset/tmdb_5000_movies.csv")
creditDf=pd.read_csv("Dataset/tmdb_5000_credits.csv")
movieDf=movieDf.rename(columns={"id":"movie_id"})
moviesDf=pd.merge(movieDf,creditDf,on="movie_id")
moviesDf=moviesDf[[
    "movie_id",
    "title_x",
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew"
]]

In [3]:

moviesDf.columns=["Id","Titles","Overview","Genres","Keywords","Cast","Crew"]

In [4]:
def convert_genres_keywords(obj):
    if pd.isna(obj):
        return []
    L = []
    for item in ast.literal_eval(obj):
        L.append(item['name'])
    return L


def get_top_cast(obj):
    if pd.isna(obj):
        return []
    L = []
    counter = 0
    for item in ast.literal_eval(obj):
        if counter < 3:
            L.append(item['name'])
            counter += 1
        else:
            break
    return L


def get_crew(obj):
    if pd.isna(obj):
        return []
    L = []
    for item in ast.literal_eval(obj):
        if item.get('job') == 'Director':
            L.append(item['name'])
            break
    return L

moviesDf['Genres'] = moviesDf['Genres'].apply(convert_genres_keywords)
moviesDf['Keywords'] = moviesDf['Keywords'].apply(convert_genres_keywords)
moviesDf['Cast'] = moviesDf['Cast'].apply(get_top_cast)
moviesDf['Crew'] = moviesDf['Crew'].apply(get_crew)

moviesDf['Overview'] = moviesDf['Overview'].fillna('')
moviesDf['Overview'] = moviesDf['Overview'].apply(lambda x: x.split())

In [5]:
def collapes_spaces(lines):
    return [i.replace(" ","") for i in lines]

moviesDf["Genres"]=moviesDf["Genres"].apply(collapes_spaces)
moviesDf["Keywords"]=moviesDf["Keywords"].apply(collapes_spaces)
moviesDf["Cast"]=moviesDf["Cast"].apply(collapes_spaces)
moviesDf["Crew"]=moviesDf["Crew"].apply(collapes_spaces)

In [6]:
moviesDf["Tags"]=moviesDf["Overview"]+moviesDf["Genres"]+moviesDf["Keywords"]+moviesDf["Cast"]+moviesDf["Crew"]
new_moviesDf=moviesDf[["Id","Titles","Tags"]].copy()
new_moviesDf["Tags"]=moviesDf["Tags"].apply(lambda s:" ".join(s).lower())

def simple_stemmer(text):
    suffixes = ('ing', 'edly', 'ed', 'ies', 'es', 's')
    stemmed_words = []
    
    for word in text.split():
        if word.endswith('ies') and len(word) > 5:
            word = word[:-3] + 'y'
        else:
            for suffix in suffixes:
                if word.endswith(suffix) and len(word) > len(suffix) + 2:
                    word = word[:-len(suffix)]
                    break
        stemmed_words.append(word)
        
    return " ".join(stemmed_words)

new_moviesDf["Tags"] = new_moviesDf["Tags"].apply(simple_stemmer)

In [7]:
cv = skl.feature_extraction.text.CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_moviesDf["Tags"]).toarray()

feature_names = cv.get_feature_names_out()
print(f"Matrix shape: {vectors.shape}")
print(f"First movie vector: {vectors[0]}")

Matrix shape: (4803, 5000)
First movie vector: [0 0 0 ... 0 0 0]


In [8]:
similarity=skl.metrics.pairwise.cosine_similarity(vectors)
distances = similarity[0]
movie_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:6]

for i in movie_list:
    print(new_moviesDf.iloc[i[0]].Titles)

Titan A.E.
Independence Day
Aliens vs Predator: Requiem
Battle: Los Angeles
Jupiter Ascending


In [9]:
def recommend(movie):
    if movie not in new_moviesDf["Titles"].values:
        print(f"Movie '{movie}' not found in dataset. Please try another Titles.")
        return
    
    movie_index = new_moviesDf[new_moviesDf["Titles"] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:6]
    print(f"Top 5 movies similar to '{movie}':")
    print("-" * 40)
    for index, (movie_idx, score) in enumerate(movies_list, start=1):
        recommended_title = new_moviesDf.iloc[movie_idx].Titles
        print(f"{index}. {recommended_title} (Similarity: {score:.3f})")

In [10]:
with open('movie_dict.pkl', 'wb') as file:
    pkl.dump(new_moviesDf.to_dict(), file)

with open('similarity.pkl', 'wb') as file:
    pkl.dump(similarity, file)